In [36]:
import pandas as pd
studentInfo = pd.read_csv("data/studentInfo.csv")
studentInfo.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [37]:
course_counts = studentInfo.groupby(
    ["code_module","code_presentation"]
).size().reset_index(name="num_students")
course_counts.sort_values("num_students",ascending=False).head(10)

,code_module,code_presentation,num_students
7,CCC,2014J,2498
18,FFF,2014J,2365
5,BBB,2014J,2292
16,FFF,2013J,2283
3,BBB,2013J,2237
9,DDD,2013J,1938
6,CCC,2014B,1936
11,DDD,2014J,1803
2,BBB,2013B,1767
15,FFF,2013B,1614


In [38]:
# Select one course
course_module = "BBB"
course_presentation = "2014J"

course_students = studentInfo[
    (studentInfo.code_module == course_module) &
    (studentInfo.code_presentation == course_presentation)
]

print("Number of students:", course_students.shape[0])
course_students.head()


Number of students: 2292


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
6365,BBB,2014J,26315,F,East Anglian Region,Lower Than A Level,60-70%,35-55,1,60,N,Pass
6366,BBB,2014J,26734,M,South West Region,A Level or Equivalent,20-30%,0-35,0,90,Y,Fail
6367,BBB,2014J,32327,F,South West Region,Lower Than A Level,20-30%,0-35,2,60,N,Withdrawn
6368,BBB,2014J,32930,F,South Region,A Level or Equivalent,40-50%,35-55,0,60,N,Pass
6369,BBB,2014J,34431,F,Yorkshire Region,Lower Than A Level,70-80%,0-35,1,60,N,Withdrawn


In [39]:
studentVle = pd.read_csv("data/studentVle.csv")
print(studentVle.shape)


(10655280, 6)


In [40]:
# Get student IDs for the selected course
bbb_student_ids = course_students["id_student"].unique()

# Filter LMS interactions
bbb_vle = studentVle[
    studentVle["id_student"].isin(bbb_student_ids)
]

print("Filtered LMS rows:", bbb_vle.shape)
bbb_vle.head()


Filtered LMS rows: (461615, 6)


,code_module,code_presentation,id_student,id_site,date,sum_click
350490,BBB,2013B,237913,542808,-9,1
350491,BBB,2013B,237913,543071,-9,2
350492,BBB,2013B,237913,543055,-9,1
350493,BBB,2013B,237913,542864,-9,10
350494,BBB,2013B,237913,543020,-9,1


In [41]:
vle = pd.read_csv("data/vle.csv")

bbb_vle = bbb_vle.merge(
    vle[["id_site", "activity_type"]],
    on="id_site",
    how="left"
)

bbb_vle.head()


,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type
0,BBB,2013B,237913,542808,-9,1,forumng
1,BBB,2013B,237913,543071,-9,2,subpage
2,BBB,2013B,237913,543055,-9,1,subpage
3,BBB,2013B,237913,542864,-9,10,homepage
4,BBB,2013B,237913,543020,-9,1,url


In [42]:
bbb_vle["week"] = (bbb_vle["date"] // 7) + 1

bbb_vle[["id_student", "date", "week", "activity_type", "sum_click"]].head()


,id_student,date,week,activity_type,sum_click
0,237913,-9,-1,forumng,1
1,237913,-9,-1,subpage,2
2,237913,-9,-1,subpage,1
3,237913,-9,-1,homepage,10
4,237913,-9,-1,url,1


In [43]:
# Keep only weeks >= 1
bbb_vle = bbb_vle[bbb_vle["week"] >= 1]

print("Min week:", bbb_vle["week"].min())
print("Max week:", bbb_vle["week"].max())


Min week: 1
Max week: 38


In [44]:
weekly_features = bbb_vle.groupby(
    ["id_student", "week", "activity_type"]
)["sum_click"].sum().unstack(fill_value=0).reset_index()

weekly_features.head()


activity_type,id_student,week,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,oucollaborate,oucontent,ouelluminate,ouwiki,page,questionnaire,quiz,resource,sharedsubpage,subpage,url
0,26315,1,0,0,0,0,39,1,33,0,32,0,0,0,0,0,13,0,24,2
1,26315,2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,26315,3,0,0,0,0,5,0,21,2,29,0,0,0,0,0,7,0,6,0
3,26315,4,0,0,0,0,14,0,3,0,0,0,0,0,0,0,0,0,0,0
4,26315,5,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [45]:
student_behavior = weekly_features.groupby("id_student", as_index=False).sum()

student_behavior.head()


activity_type,id_student,week,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,oucollaborate,oucontent,ouelluminate,ouwiki,page,questionnaire,quiz,resource,sharedsubpage,subpage,url
0,26315,636,0,0,0,0,255,3,393,13,905,0,0,0,14,501,89,0,135,6
1,26734,130,0,0,0,0,16,1,63,19,53,0,0,0,0,5,30,0,15,1
2,32327,97,0,0,0,0,135,0,98,8,114,0,0,0,0,55,10,0,7,0
3,32930,203,0,0,0,0,0,0,96,0,151,0,0,0,0,152,21,0,16,2
4,34431,40,0,0,0,0,104,0,54,0,0,0,0,0,0,16,1,0,8,9


In [46]:
# Drop week column before student-level aggregation
weekly_no_week = weekly_features.drop(columns=["week"])

student_behavior = weekly_no_week.groupby("id_student", as_index=False).sum()

student_behavior.head()




activity_type,id_student,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,oucollaborate,oucontent,ouelluminate,ouwiki,page,questionnaire,quiz,resource,sharedsubpage,subpage,url
0,26315,0,0,0,0,255,3,393,13,905,0,0,0,14,501,89,0,135,6
1,26734,0,0,0,0,16,1,63,19,53,0,0,0,0,5,30,0,15,1
2,32327,0,0,0,0,135,0,98,8,114,0,0,0,0,55,10,0,7,0
3,32930,0,0,0,0,0,0,96,0,151,0,0,0,0,152,21,0,16,2
4,34431,0,0,0,0,104,0,54,0,0,0,0,0,0,16,1,0,8,9


In [47]:
student_behavior.shape

(1889, 19)

In [48]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = student_behavior.drop(columns=["id_student"])
X_scaled = scaler.fit_transform(X)



In [49]:
student_behavior.columns

Index(['id_student', 'dataplus', 'dualpane', 'externalquiz', 'folder',
       'forumng', 'glossary', 'homepage', 'oucollaborate', 'oucontent',
       'ouelluminate', 'ouwiki', 'page', 'questionnaire', 'quiz', 'resource',
       'sharedsubpage', 'subpage', 'url'],
      dtype='str', name='activity_type')

In [50]:
X.shape

(1889, 18)

In [51]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sil_scores = {}

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)

sil_scores


{2: 0.4596676908302521,
 3: 0.4585535306492954,
 4: 0.45420819754154523,
 5: 0.45427946594068047,
 6: 0.4564866531831848,
 7: 0.4132663479345377}

In [52]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
student_behavior["cluster"] = kmeans.fit_predict(X_scaled)

student_behavior["cluster"].value_counts()


cluster
1    1367
0     520
2       1
3       1
Name: count, dtype: int64

In [53]:
# Compute total activity per student
student_behavior["total_activity"] = student_behavior.sum(axis=1)

student_behavior["total_activity"].describe()


count    1.889000e+03
mean     7.213196e+05
std      4.778661e+05
min      2.693800e+04
25%      5.839030e+05
50%      6.504050e+05
75%      6.881950e+05
max      2.699295e+06
Name: total_activity, dtype: float64

In [54]:
threshold = student_behavior["total_activity"].quantile(0.99)

student_behavior_clean = student_behavior[
    student_behavior["total_activity"] <= threshold
].drop(columns=["total_activity"])

student_behavior_clean.shape


(1870, 20)

In [55]:
import numpy as np

student_behavior_log = np.log1p(student_behavior_clean)


In [56]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(student_behavior_log)


In [57]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
student_behavior_clean["cluster"] = kmeans.fit_predict(X_scaled)

student_behavior_clean["cluster"].value_counts()


cluster
1    967
0    901
2      1
3      1
Name: count, dtype: int64

In [58]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
student_behavior_clean["cluster"] = kmeans.fit_predict(X_scaled)

student_behavior_clean["cluster"].value_counts()


cluster
1    968
0    901
2      1
Name: count, dtype: int64

In [59]:
student_behavior_clean.groupby("cluster").mean()


activity_type,id_student,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,oucollaborate,oucontent,ouelluminate,ouwiki,page,questionnaire,quiz,resource,sharedsubpage,subpage,url
cluster,,,,,,,,,,,,,,,,,,,
0,691753.260821,0.000000,0.0,0.000000,0.000000,16.736959,0.442841,52.871254,1.135405,117.068812,0.000000,0.000000,0.000000,0.250832,28.415094,15.447281,0.000000,12.472808,1.042175
1,708960.287190,0.004132,0.0,0.024793,0.003099,180.864669,4.607438,248.454545,10.253099,744.391529,0.011364,0.238636,0.012397,6.653926,153.609504,56.481405,0.004132,48.148760,2.978306
2,557085.000000,14.000000,2.0,0.000000,0.000000,716.000000,7.000000,1123.000000,7.000000,2990.000000,0.000000,17.000000,3.000000,26.000000,2733.000000,68.000000,0.000000,503.000000,33.000000


In [60]:
# Remove tiny outlier cluster
student_behavior_final = student_behavior_clean[
    student_behavior_clean["cluster"] != 1
]

student_behavior_final["cluster"].value_counts()


cluster
0    901
2      1
Name: count, dtype: int64

In [61]:
student_behavior_final["cluster_label"] = student_behavior_final["cluster"].map({
    0: "Moderate Engagement",
    2: "High Engagement"
})
import os

os.makedirs("outputs", exist_ok=True)

student_behavior_final.to_csv(
    "outputs/clustered_students.csv" ,
    index=False
)


In [62]:
### Phase 1 Complete
# - Course: BBB–2014J
# - Active students: 889
# - Clusters identified: 2 (High vs Moderate engagement)
# - Outliers removed: Yes


In [63]:
import os

os.getcwd()

'd:\\minor project code'

In [64]:
import os

print(os.listdir("outputs"))


['clustered_students.csv']


In [65]:
import pandas as pd

pd.read_csv("outputs/clustered_students.csv").head()


,id_student,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,oucollaborate,oucontent,...,ouwiki,page,questionnaire,quiz,resource,sharedsubpage,subpage,url,cluster,cluster_label
0,26734,0,0,0,0,16,1,63,19,53,...,0,0,0,5,30,0,15,1,0,Moderate Engagement
1,32327,0,0,0,0,135,0,98,8,114,...,0,0,0,55,10,0,7,0,0,Moderate Engagement
2,32930,0,0,0,0,0,0,96,0,151,...,0,0,0,152,21,0,16,2,0,Moderate Engagement
3,34431,0,0,0,0,104,0,54,0,0,...,0,0,0,16,1,0,8,9,0,Moderate Engagement
4,34662,0,0,0,0,14,0,106,2,62,...,0,0,0,0,20,0,8,2,0,Moderate Engagement
